# MUSEmotion Shareable Colab

This notebook is designed to be opened directly from GitHub in Google Colab. It first runs the lightweight real-trained checkpoints committed in `models/real_training/`, so viewers can generate a MIDI sample without retraining. Optional cells at the end show how to retrain the smoke pipeline on Colab.

## 1. Clone The GitHub Repository And Install Dependencies

In [ ]:
!test -d /content/ECE1508-Deep-Generative-Models || git clone https://github.com/Alex-Xi-Chen/ECE1508-Deep-Generative-Models.git /content/ECE1508-Deep-Generative-Models
%cd /content/ECE1508-Deep-Generative-Models
!git pull --ff-only
!pip install -q -r requirements.txt
!pip install -q "transformers==4.44.2" "huggingface-hub<1.0" "tokenizers>=0.19,<0.20"
!pip install -q -e . --no-deps

## 2. Check Runtime

In [ ]:
import subprocess
import torch

print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime')
if torch.cuda.is_available():
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))

## 3. Verify The Committed Real-Trained Models

In [ ]:
from pathlib import Path

required_paths = [
    Path('models/real_training/classifier_tiny_bert_42epoch/pytorch_model.bin'),
    Path('models/real_training/classifier_tiny_bert_42epoch/config.json'),
    Path('models/real_training/classifier_tiny_bert_42epoch/vocab.txt'),
    Path('models/real_training/music_transformer_42epoch/best.pt'),
    Path('models/real_training/music_transformer_42epoch/tokenizer/vocab.json'),
]

for path in required_paths:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

assert all(path.exists() for path in required_paths), 'Missing committed model artifact. Make sure the notebook is running from the latest main branch.'

## 4. Generate A MIDI Clip From The GitHub Model

The default `configs/inference.yaml` points at `models/real_training/`, not ignored `artifacts/` paths.

In [ ]:
!python -m musemotion.cli.generate \
  --config configs/inference.yaml \
  --text "I feel hopeful but calm today" \
  --output artifacts/samples/shareable_colab_demo.mid \
  --seed 1508 \
  --max-tokens 128

In [ ]:
from pathlib import Path
from google.colab import files

midi_path = Path('artifacts/samples/shareable_colab_demo.mid')
print('generated_midi', midi_path.resolve())
print('size_bytes', midi_path.stat().st_size)
files.download(str(midi_path))

## 5. Optional: Launch A Shareable Gradio Demo

In [ ]:
!python -m musemotion.frontend.app --config configs/inference.yaml --share

## Optional: Re-Train The Smoke Pipeline On Colab

These cells download EMOPIA and run the capped smoke configs. They are useful for demonstrating the full training flow, but they are not required for generation from the committed model above.

In [ ]:
!python -m musemotion.cli.download_emopia --output data/raw/emopia

In [ ]:
!python -m musemotion.cli.train_classifier --config configs/colab_smoke_classifier.yaml

In [ ]:
!python -m musemotion.cli.prepare_emopia --config configs/colab_smoke_music.yaml

In [ ]:
!python -m musemotion.cli.train_generator --config configs/colab_smoke_music.yaml

In [ ]:
!python -m musemotion.cli.generate \
  --config configs/inference_colab_smoke.yaml \
  --text "I feel hopeful but calm today" \
  --output artifacts/colab_smoke/samples/hopeful_calm.mid \
  --seed 1508